<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_16_practicum_hashing/note_lesson_16_hashing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 16 — Практикум П3. Хеш-структури: третя зміна диспетчера таксі

У П1 диспетчерська навчилася рахувати ціну рішення, у П2 — обирати стратегію за властивостями даних. Сьогодні журнал поїздок приходить **у порядку дзвінків, невідсортованим**. Замість чекати зручних даних змінимо **представлення**: один раз перекладемо дані у `dict` / `set` / `Counter` — і далі кожна відповідь займе один крок.

Виконуй клітинки **зверху вниз**. Перед клітинками з позначкою **Прогноз** спершу скажи, що буде. Теорія — у книзі: [Урок 16. Практикум П3. Хеш-структури](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m1/lesson_16/).

## 🔁 Пригадай (без підглядання)

1. На яку властивість списку спиралися два вказівники в П2?
2. Яка складність `x in список` і `x in множина`?
3. Що робить `counts[name] = counts.get(name, 0) + 1`?

<details>
<summary>Відповіді</summary>

1. На відсортованість.
2. `O(n)` і `O(1)` в середньому.
3. Збільшує лічильник `name` на 1; для нового ключа `get` дає 0.

</details>

## 0. Журнал зміни

In [ ]:
from typing import NamedTuple


class Trip(NamedTuple):
    id: int
    client: str
    start: str
    end: str
    fare: int


trips = [
    Trip(1120, "Марта", "Поділ", "Оболонь", 230),
    Trip(425, "Олег", "Вокзал", "Поділ", 150),
    Trip(930, "Марта", "Оболонь", "Поділ", 270),
    Trip(612, "Ірина", "Печерськ", "Вокзал", 180),
    Trip(845, "Тарас", "Поділ", "Оболонь", 320),
    Trip(1035, "Олег", "Поділ", "Вокзал", 120),
    Trip(700, "Олена", "Оболонь", "Поділ", 410),
    Trip(510, "Богдан", "Вокзал", "Печерськ", 150),
]

fares = [trip.fare for trip in trips]
print(fares)

## 1. Ваучер на 500 грн: два вказівники з П2

**Прогноз:** що поверне `pair_with_sum` на невідсортованому списку? (`230 + 270 = 500`)

In [ ]:
def pair_with_sum(items, target):
    left, right = 0, len(items) - 1
    steps = 0
    while left < right:
        steps += 1
        total = items[left] + items[right]
        if total == target:
            return (items[left], items[right]), steps
        if total < target:
            left += 1
        else:
            right -= 1
    return None, steps


print(pair_with_sum(fares, 500))

<details>
<summary>Відповідь</summary>

`(None, 7)` — «пари немає». Жодної помилки: алгоритм мовчки спирається на порядок, якого немає.

</details>

In [ ]:
def pair_with_sum_brute(items, target):
    steps = 0
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            steps += 1
            if items[i] + items[j] == target:
                return (i, j), steps
    return None, steps


print(pair_with_sum_brute(fares, 500))

## 🛠 Вправа 1. Two Sum за один прохід

Напиши `pair_with_sum_hash(items, target)`: словник `seen` «сума → позиція»; для кожної суми питаємо, чи бачили **доповнення** `target - value`. Повертає `((позиція1, позиція2), кроки)` або `(None, кроки)`.

In [ ]:
def pair_with_sum_hash(items, target):
    seen = {}
    steps = 0
    # YOUR CODE HERE
    # BEGIN SOLUTION
    for i, value in enumerate(items):
        steps += 1
        need = target - value
        if need in seen:
            return (seen[need], i), steps
        seen[value] = i
    # END SOLUTION
    return None, steps


print(pair_with_sum_hash(fares, 500))
assert pair_with_sum_hash(fares, 500) == ((0, 2), 3)
assert pair_with_sum_hash(fares, 10_000) == (None, 8)
assert pair_with_sum_hash([], 500) == (None, 0)
print("✅ Вправа 1 пройдена")

### Дослід подвоєння

**Прогноз:** у скільки разів зростуть кроки перебору і словника, коли журнал подвоюється?

In [ ]:
for n in [1000, 2000, 4000]:
    values = list(range(1, n + 1))
    _, brute = pair_with_sum_brute(values, 10**9)
    _, fast = pair_with_sum_hash(values, 10**9)
    print(n, brute, fast)

<details>
<summary>Відповідь</summary>

Перебір — ×4 (`O(n²)`), словник — ×2 (`O(n)`).

</details>

## 2. Поїздка за номером

Лінійний пошук на кожен дзвінок проти словника-індексу, побудованого один раз.

In [ ]:
def find_trip(trips, trip_id):
    steps = 0
    for trip in trips:
        steps += 1
        if trip.id == trip_id:
            return trip, steps
    return None, steps


print(find_trip(trips, 845))

## 🛠 Вправа 2. Словник-індекс

Побудуй `by_id` — словник «номер поїздки → поїздка» одним dict comprehension.

In [ ]:
# YOUR CODE HERE
# BEGIN SOLUTION
by_id = {trip.id: trip for trip in trips}
# END SOLUTION

print(by_id[845].client)
assert by_id[845].client == "Тарас"
assert by_id.get(999) is None
assert len(by_id) == len(trips)
print("✅ Вправа 2 пройдена")

**Прогноз:** скільки кроків на 100 дзвінків у журналі з 1000 поїздок — лінійно й через індекс?

In [ ]:
import random

ids = list(range(1000))
random.Random(7).shuffle(ids)
big_journal = [Trip(i, "клієнт", "А", "Б", 100) for i in ids]
calls = list(range(0, 1000, 10))

linear_steps = sum(find_trip(big_journal, trip_id)[1] for trip_id in calls)
index = {trip.id: trip for trip in big_journal}
index_steps = len(big_journal) + len(calls)

print(linear_steps, index_steps)

<details>
<summary>Відповідь</summary>

Лінійно — понад 53 000 (у середньому пів журналу на дзвінок), з індексом — 1000 на побудову + 100 запитів. Для **одного** дзвінка індекс не окупився б.

</details>

## 3. Як словник знаходить ключ: кошики

Кошик = `hash(ключ) % кількість_кошиків`. Для малих цілих `hash(n) == n`.

In [ ]:
buckets = [[] for _ in range(8)]
for trip_id in [1120, 425, 930, 612, 845]:
    buckets[hash(trip_id) % 8].append(trip_id)

for number, bucket in enumerate(buckets):
    print(number, bucket)

**Прогноз:** у який кошик потрапить 1128 і скільки порівнянь знадобиться, щоб його знайти?

In [ ]:
def find_in_table(buckets, key):
    bucket = buckets[hash(key) % len(buckets)]
    comparisons = 0
    for item in bucket:
        comparisons += 1
        if item == key:
            return True, comparisons
    return False, comparisons


buckets[hash(1128) % 8].append(1128)
print(buckets[0])
print(find_in_table(buckets, 1128))
print(find_in_table(buckets, 845))

<details>
<summary>Відповідь</summary>

`1128 % 8 = 0` — кошик 0, де вже є 1120: **колізія**, 2 порівняння. Справжній `dict` тримає кошиків більше, ніж ключів, тож колізій мало і пошук в середньому `O(1)`.

</details>

## 4. Що може бути ключем

Лише **hashable** — незмінні значення. Розкоментуй другий рядок: `TypeError: unhashable type: 'list'`. Потім закоментуй назад.

In [ ]:
route_counts = {("Поділ", "Оболонь"): 2}          # кортеж — можна
# route_counts[["Поділ", "Оболонь"]] = 2          # список — не можна
print(route_counts[("Поділ", "Оболонь")])

## 5. Найпопулярніший маршрут

**Прогноз:** які маршрути та скільки разів порахує наївний лічильник?

In [ ]:
def route_counts_naive(trips):
    counts = {}
    for trip in trips:
        route = (trip.start, trip.end)
        counts[route] = counts.get(route, 0) + 1
    return counts


print(route_counts_naive(trips))

<details>
<summary>Відповідь</summary>

«Поділ → Оболонь» і «Оболонь → Поділ» — окремими ключами по 2. Для диспетчера це один маршрут: потрібен канонічний ключ.

</details>

## 🛠 Вправа 3. Канонічний ключ маршруту

`route_key(trip)` — відсортований кортеж з двох точок, щоб напрямок не мав значення.

In [ ]:
from collections import Counter


def route_key(trip):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return tuple(sorted((trip.start, trip.end)))
    # END SOLUTION


routes = Counter(route_key(trip) for trip in trips)
print(routes.most_common(2))
assert route_key(trips[0]) == route_key(trips[2]) == ("Оболонь", "Поділ")
assert routes.most_common(1) == [(("Оболонь", "Поділ"), 4)]
print("✅ Вправа 3 пройдена")

## 6. Пастка: словник змінюється під час обходу

Розкоментуй цикл — `RuntimeError: dictionary changed size during iteration`. Правильно — будувати новий словник.

In [ ]:
clients = Counter(trip.client for trip in trips)
# for name in clients:
#     if clients[name] < 2:
#         del clients[name]
regular = {name: count for name, count in clients.items() if count >= 2}
print(regular)

## 7. Розібраний приклад: звіт диспетчера за зміну

In [ ]:
def shift_report(trips, voucher):
    pair, _ = pair_with_sum_hash([trip.fare for trip in trips], voucher)
    voucher_trips = [trips[i].id for i in pair] if pair else None
    routes = Counter(route_key(trip) for trip in trips)
    top_route, top_count = routes.most_common(1)[0]
    clients = Counter(trip.client for trip in trips)
    return {
        "voucher_trips": voucher_trips,
        "top_route": " — ".join(top_route),
        "top_route_trips": top_count,
        "regular_clients": [name for name, count in clients.items() if count >= 2],
    }


report = shift_report(trips, 500)
for key, value in report.items():
    print(key, value)

## 🛠 Вправа 4. Перший разовий клієнт

`first_unique(names)` — перше за журналом ім'я, яке трапляється рівно один раз, або `None`. Два проходи: `Counter`, потім пошук по **списку**. Без `names.count(...)` у циклі.

In [ ]:
def first_unique(names):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    counts = Counter(names)
    for name in names:
        if counts[name] == 1:
            return name
    return None
    # END SOLUTION


print(first_unique([trip.client for trip in trips]))
assert first_unique([trip.client for trip in trips]) == "Ірина"
assert first_unique(["Марта", "Марта"]) is None
assert first_unique([]) is None
print("✅ Вправа 4 пройдена")

## 🛠 Вправа 5. Анаграми — класика співбесід

- `is_anagram(a, b)` — `Counter` літер, без урахування регістру;
- `group_anagrams(words)` — групи в порядку першої появи, ключ — `"".join(sorted(word.lower()))`, один прохід.

In [ ]:
def is_anagram(a, b):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return Counter(a.lower()) == Counter(b.lower())
    # END SOLUTION


def group_anagrams(words):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    groups = {}
    for word in words:
        key = "".join(sorted(word.lower()))
        groups.setdefault(key, []).append(word)
    return list(groups.values())
    # END SOLUTION


assert is_anagram("Літо", "тіло") and not is_anagram("літо", "літа")
words = ["літо", "клоун", "тіло", "таксі", "уклон", "кит"]
print(group_anagrams(words))
assert group_anagrams(words) == [["літо", "тіло"], ["клоун", "уклон"], ["таксі"], ["кит"]]
assert group_anagrams([]) == []
print("✅ Вправа 5 пройдена")

## ✅ Самоперевірка

1. Чому два вказівники мовчки помиляються на невідсортованих даних?
2. Що зберігає `seen` у Two Sum і чому словник, а не список?
3. Коли словник-індекс не окупається?
4. Чому список не може бути ключем, а кортеж — може?
5. Навіщо канонічний ключ маршруту?

<details>
<summary>Відповіді</summary>

1. Алгоритм не перевіряє порядок — лише покладається на нього.
2. Побачені суми та позиції; перевірка «чи бачили» в словнику — один крок, у списку — прохід.
3. Коли запит один або їх мало: побудова — n кроків, лінійний пошук — у середньому n / 2.
4. Список змінюється, його хеш став би іншим; кортеж з незмінних елементів — ні.
5. Щоб різні напрямки одного маршруту потрапили в один лічильник.

</details>

### Шпаргалка

```python
seen = {}                                        # Two Sum: значення → позиція
if target - value in seen: ...                   # O(1) в середньому
by_id = {trip.id: trip for trip in trips}        # індекс: O(n) один раз, далі O(1)
key = tuple(sorted((a, b)))                      # канонічний ключ
Counter(items).most_common(3)                    # частоти
{k: v for k, v in d.items() if v >= 2}           # не видаляй під час обходу — будуй новий
# ключі dict і елементи set — лише hashable: str, int, tuple; list — TypeError
```

## Далі

**Урок 17 — огляд модуля 1**: усе разом — типи, колекції, функції, винятки, файли, Git і три практикуми.